# Build the CryoSat-2 interpolation-window master file

This notebook uses daily GNSS positions, corrected GNSS heights, the REMA variogram parameters, and the MISWIS parquet files. It writes the window master CSV, point subsets, and window diagnostics.

Run it from the `manuscript_scripts` folder so the bundled `data/corrected_timeseries/` path resolves correctly. Update the machine-specific input and output paths in the settings cell before running it on another computer.


## Imports and settings

Set the paths and station/window options here before running the workflow.


In [ ]:
from __future__ import annotations
# ======================================================================
#  Preprocessing for R::fields workflow
#  - Per-station iterative clipping (sigma0_r1, h_r1)
#  - 90-day windows for all stations
#  - Writes:
#       * kriging_windows_master.csv
#       * one CS2-points CSV per window
#       * per-window point maps (elevation + temporal)
#       * per-station diagnostics of kept / canceled windows
# ======================================================================

import os
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from pyproj import Transformer
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Circle

# ---------------------------
# CONFIG
# ---------------------------
# Update these locations to the MISWIS parquet data, daily GNSS .pos files,
# and output folder. FILTERED_GNSS points to this repository's corrected-time-series CSVs.

PARQUET_CS2   = "/Volumes/icefall/marquardt/data/CS2/miswis"
SCRIPT_DIR    = Path.cwd()
FILTERED_GNSS = os.path.join(SCRIPT_DIR, "data", "corrected_timeseries")
DAILY_POS_DIR = "/Volumes/icefall/tinydrive/GNSS_data/daily_positions"
OUTPUT_DIR    = "/Volumes/icefall/marquardt/results/bias_results"

# REMA variogram CSV exported by estimate_rema_variogram_parameters_from_gnss_mid_dates.R.
# It must contain station-level aRange_m and r02_m values.
VARIOGRAM_PARAMS_CSV = os.path.join(OUTPUT_DIR, "rema_variogram_params_station_midpoint.csv")

# Stations of interest
STATIONS = [
    # The 29 stations listed in the manuscript (GZ16 is excluded).
    "gz01", "gz02", "gz06", "gz09", "gz13", "gz17", "gz18",
    "la01", "la02", "la03", "la04", "la05", "la06", "la07", "la08",
    "la09", "la10", "la11", "la12", "la13", "la14", "la15", "la16",
    "la17", "la18", "mg04", "mg05", "mg06", "mg07"
]

# 90-day window, with 1/3 overlap (30 days)
TIME_WINDOW_DAYS = 90
STEP_DAYS        = 30

# discard windows whose center is before this date
POST_DATE_CUTOFF = pd.Timestamp("2011-01-01")

# small helper cutoff for bad LA05 daily positions
POS_CUTOFFS = {
    "LA05": pd.Timestamp("2010-12-07")
}

# ---------------------------


## Helper functions

These functions load the inputs, define the time windows, and calculate the GNSS and CryoSat-2 window metrics.


In [ ]:
# small helpers
# ---------------------------

def safe_makedirs(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path

_tf = Transformer.from_crs("EPSG:4326", "EPSG:3031", always_xy=True)
ll2xy = _tf.transform

def _read_parquet_any(path: str) -> pd.DataFrame:
    last_err = None
    for eng in ("pyarrow", "fastparquet"):
        try:
            print(f"[DEBUG] Reading parquet with engine={eng}: {path}")
            df = pd.read_parquet(path, engine=eng)
            print(f"[DEBUG]   read {len(df)} rows, columns={list(df.columns)}")
            return df
        except Exception as e:
            print(f"[DEBUG]   engine={eng} failed: {e}")
            last_err = e
    raise ImportError("Install a Parquet engine: pyarrow or fastparquet") from last_err

def load_cs2(parquet_dir: str) -> gpd.GeoDataFrame:
    print(f"[DEBUG] Loading CS2 parquet files from: {parquet_dir}")
    files = glob.glob(os.path.join(parquet_dir, "*.parquet"))
    if not files:
        raise FileNotFoundError(f"No parquet files in {parquet_dir}")
    print(f"[DEBUG]   found {len(files)} parquet files")

    lst = []
    for f in files:
        df = _read_parquet_any(f)
        lst.append(df)

    cs2 = pd.concat(lst, ignore_index=True)

    # drop bad row if present
    if 7386780 in cs2.index:
        print("[DEBUG] Dropping CS2 row with index 7386780 (bad row)")
        cs2 = cs2.drop(index=7386780)

    # convert time if needed
    if not pd.api.types.is_datetime64_any_dtype(cs2["time"]):
        print("[DEBUG] Converting CS2 'time' from seconds since 2000-01-01 to datetime")
        cs2["time"] = pd.to_datetime(cs2["time"], unit="s", origin="2000-01-01")

    print(f"[DEBUG] CS2 time range: {cs2['time'].min()} → {cs2['time'].max()}")
    gdf = gpd.GeoDataFrame(cs2, geometry=gpd.points_from_xy(cs2.x, cs2.y), crs="EPSG:3031")

    # store row ID explicitly
    gdf["cs2_row"] = gdf.index
    return gdf

def overlapping_windows(tmin: pd.Timestamp,
                        tmax: pd.Timestamp,
                        window_days: int,
                        step_days: int):
    if pd.isna(tmin) or pd.isna(tmax) or tmin >= tmax or window_days <= 0 or step_days <= 0:
        return []
    half = pd.Timedelta(days=window_days // 2)
    mids = pd.date_range(tmin + half, tmax - half, freq=f"{step_days}D")
    return [(m - half, m + half, m) for m in mids]

# candidate lat/lon columns in daily positions
LAT_CANDIDATES = ["lat_daily", "lat_mean", "lat_deg", "lat", "Latitude"]
LON_CANDIDATES = ["lon_daily", "lon_mean", "lon_deg", "lon", "Longitude"]

POSITION_ALIAS = {"la02": "la09", "la09": "la02"}
GNSS_ALIAS     = {"la02": "la09", "la09": "la02"}

def find_latlon_cols(df: pd.DataFrame):
    lat_col = next((c for c in LAT_CANDIDATES if c in df.columns), None)
    lon_col = next((c for c in LON_CANDIDATES if c in df.columns), None)
    return lat_col, lon_col

def _align_lon_to_ref(lon_series: pd.Series, ref_lon: float):
    x = pd.to_numeric(lon_series, errors="coerce").to_numpy(dtype=float)
    return ref_lon + ((x - ref_lon + 180.0) % 360.0) - 180.0

def load_daily_positions(station: str) -> pd.DataFrame | None:
    src = POSITION_ALIAS.get(station.lower(), station.lower())
    pth = os.path.join(DAILY_POS_DIR, f"{src}_daily_mean_2011_2026.csv")
    print(f"[DEBUG] Loading daily positions for {station} from: {pth}")
    if not os.path.exists(pth):
        print(f"[DEBUG]   daily positions file missing: {pth}")
        return None

    df = pd.read_csv(pth)
    print(f"[DEBUG]   read {len(df)} rows, columns={list(df.columns)}")

    if "date" in df.columns:
        df["date"] = pd.to_datetime(df["date"])
        df = df.set_index("date")
    else:
        df.index = pd.to_datetime(df.index)

    stU = station.upper()
    if stU in POS_CUTOFFS:
        print(f"[DEBUG]   applying POS_CUTOFF for {stU}: {POS_CUTOFFS[stU]}")
        df = df.loc[df.index >= POS_CUTOFFS[stU]].copy()

    lat_col, lon_col = find_latlon_cols(df)
    print(f"[DEBUG]   lat_col={lat_col}, lon_col={lon_col}")

    if lon_col is not None:
        df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
        wrap = df[lon_col] > 180
        if wrap.any():
            print(f"[DEBUG]   wrapping {wrap.sum()} longitudes > 180 to [-180, 180]")
            df.loc[wrap, lon_col] = df.loc[wrap, lon_col] - 360.0

    return df

def window_avg_gnss_location(pos_df: pd.DataFrame,
                             t_center: pd.Timestamp,
                             half_days: int,
                             method: str = "mean",
                             valid_dates=None):
    if pos_df is None or pos_df.empty:
        return {"available": False}

    lat_col, lon_col = find_latlon_cols(pos_df)
    if lat_col is None or lon_col is None:
        return {"available": False}

    half = pd.Timedelta(days=half_days)
    sub = pos_df.loc[
        (pos_df.index >= t_center - half) &
        (pos_df.index <= t_center + half),
        [lat_col, lon_col]
    ].dropna()

    if valid_dates is not None:
        if not isinstance(valid_dates, pd.DatetimeIndex):
            valid_dates = pd.to_datetime(pd.Index(valid_dates))
        sub = sub.loc[sub.index.isin(valid_dates)]

    if sub.empty:
        return {"available": False}

    lat_vals = pd.to_numeric(sub[lat_col], errors="coerce").to_numpy(float)
    lon_vals = _align_lon_to_ref(sub[lon_col], float(sub[lon_col].iloc[0]))

    if method == "median":
        lat_w = float(np.nanmedian(lat_vals))
        lon_w = float(np.nanmedian(lon_vals))
    else:
        lat_w = float(np.nanmean(lat_vals))
        lon_w = float(np.nanmean(lon_vals))

    lon_w = ((lon_w + 180.0) % 360.0) - 180.0
    sx_w, sy_w = ll2xy(lon_w, lat_w)
    span_days = float((sub.index.max() - sub.index.min()).days)

    return {
        "available": True,
        "lat_w": lat_w,
        "lon_w": lon_w,
        "sx_w": sx_w,
        "sy_w": sy_w,
        "n": int(len(sub)),
        "span_days": span_days,
        "source": f"window_{method}_restricted_to_GNSS_dates"
    }

def load_gnss_heights(station: str) -> pd.DataFrame | None:
    src = GNSS_ALIAS.get(station.lower(), station.lower()).lower()
    pth = os.path.join(FILTERED_GNSS, f"{src}_classified_rh_apc_corrected.csv")
    if not os.path.isfile(pth):
        print(f"[DEBUG]   GNSS file missing for {station}: {pth}")
        return None

    print(f"[DEBUG] Loading GNSS heights for {station} from: {pth}")
    df = pd.read_csv(pth, parse_dates=["YEAR-MM-DD"])
    print(f"[DEBUG]   read {len(df)} GNSS rows, columns={list(df.columns)}")

    df = df[df["YEAR-MM-DD"] >= "2011-01-01"].copy()
    df = df.rename(columns={
        "Adjusted(m)": "HGT_ADJUSTED(m)",
        "GNSS std": "HGT_ADJUSTED_uncertainty"
    })
    if "HGT_ADJUSTED_uncertainty" not in df.columns:
        df["HGT_ADJUSTED_uncertainty"] = np.nan

    return df.set_index("YEAR-MM-DD").sort_index()

def iterative_sigma_clip(df: pd.DataFrame,
                         col: str = "h_r1",
                         z: float = 3.0,
                         max_delta_pct: float = 5.0) -> pd.DataFrame:
    cur = df.copy()
    prev_n = len(cur)
    it = 0
    print(f"[DEBUG] Starting iterative sigma clip for station on '{col}' "
          f"with N={prev_n}, z={z}, max_delta_pct={max_delta_pct}")
    while True:
        it += 1
        mu = cur[col].mean()
        sd = cur[col].std()
        print(f"[DEBUG]   iter {it}: mean={mu:.6f}, sd={sd:.6f}, N={len(cur)}")

        if not np.isfinite(mu) or not np.isfinite(sd) or sd == 0:
            print(f"[DEBUG]   [{col}] iter {it}: degenerate stats; stop.")
            break

        keep = cur[col].between(mu - z * sd, mu + z * sd)
        new = cur.loc[keep]
        removed_pct = 100.0 * (len(cur) - len(new)) / max(1, len(cur))
        print(f"[DEBUG]   [{col}] iter {it}: kept {len(new):,} (removed {removed_pct:.2f}%)")

        cur = new
        if removed_pct < max_delta_pct or len(cur) == prev_n:
            print(f"[DEBUG]   [{col}] stopping iter clip at iter {it}")
            break
        prev_n = len(cur)
    return cur


## Load shared input data


In [ ]:
# ---------------------------
# LOAD SHARED DATA
# ---------------------------

safe_makedirs(OUTPUT_DIR)

cs2_gd = load_cs2(PARQUET_CS2)

print(f"[DEBUG] Loading REMA variogram parameters from: {VARIOGRAM_PARAMS_CSV}")
variogram_df = pd.read_csv(VARIOGRAM_PARAMS_CSV)
variogram_df["station"] = variogram_df["station"].astype(str).str.upper()
if not {"station", "r02_m"}.issubset(variogram_df.columns):
    raise ValueError(f"{VARIOGRAM_PARAMS_CSV} must contain ['station','r02_m']")
variogram_df["r02_m"] = pd.to_numeric(variogram_df["r02_m"], errors="coerce")
r02_by_station = dict(zip(variogram_df["station"], variogram_df["r02_m"]))
missing_r02 = [st.upper() for st in STATIONS if st.upper() not in r02_by_station]
invalid_r02 = [st for st in STATIONS if not np.isfinite(r02_by_station.get(st.upper(), np.nan)) or r02_by_station[st.upper()] <= 0]
if missing_r02 or invalid_r02:
    raise ValueError(
        f"Missing or invalid r02_m values for stations: {sorted(set(missing_r02 + invalid_r02))}"
    )
print(f"[DEBUG]   loaded valid r02_m values for {len(r02_by_station)} stations")

# Output locations
WINDOW_POINTS_ROOT = safe_makedirs(os.path.join(OUTPUT_DIR, "kriging_window_points"))
KRIG_META_CSV      = os.path.join(OUTPUT_DIR, "kriging_windows_master.csv")



## Build station windows and write point subsets

This cell loops over the 29 paper stations, applies the existing station-level clipping, and writes the per-window point files and diagnostics.


In [ ]:
# ---------------------------
# MAIN LOOP – per-station clipping
# ---------------------------

master_records = []

for station in STATIONS:
    st = station.upper()
    print(f"\n===== STATION {st} =====")

    # per-station diagnostics
    window_status_records: list[dict] = []
    win_status_index: dict[str, int] = {}

    st_root_dir = safe_makedirs(os.path.join(OUTPUT_DIR, st))

    gnss = load_gnss_heights(st)
    if gnss is None or gnss.empty:
        print(f"  ⚠️ {st}: no adjusted GNSS record; skipping.")
        continue

    valid_gnss = gnss.loc[gnss["HGT_ADJUSTED(m)"].notna()]
    if valid_gnss.empty:
        print(f"  ⚠️ {st}: no valid adjusted GNSS heights; skipping.")
        continue

    # Use the first and last dates with adjusted GNSS heights as the station range.
    t0 = valid_gnss.index.min()
    t1 = valid_gnss.index.max()
    print(f"  adjusted GNSS time range: {t0} → {t1}")

    # CS2 subset within the adjusted GNSS record range
    cs2_sta = cs2_gd[(cs2_gd["time"] >= t0) & (cs2_gd["time"] <= t1)].copy()
    print(f"  CS2 rows in adjusted GNSS range: {len(cs2_sta)}")
    if cs2_sta.empty:
        print(f"  ⚠️ {st}: no CS2 in adjusted GNSS range; skipping.")
        continue

    pos_df = load_daily_positions(st)
    if pos_df is None or pos_df.empty:
        print(f"  ⚠️ {st}: missing daily positions; skipping.")
        continue

    # 90-day overlapping windows over CS2 time range
    wins_ov = overlapping_windows(
        cs2_sta["time"].min(), cs2_sta["time"].max(),
        window_days=TIME_WINDOW_DAYS,
        step_days=STEP_DAYS
    )
    print(f"  overlapping windows count: {len(wins_ov)}")
    if not wins_ov:
        continue

    half_days = TIME_WINDOW_DAYS // 2

    # ---- FIRST PASS: gather candidate windows + build station-level pool ----
    station_pool_chunks = []    # for iterative clipping per station
    window_candidates   = []    # store indices per window before clipping

    r02_used_global = float(r02_by_station[st])
    print(f"  using REMA r02 search radius R={r02_used_global:.1f} m")

    for (w0, w1, mid) in wins_ov:
        w0 = pd.to_datetime(w0)
        w1 = pd.to_datetime(w1)
        mid = pd.to_datetime(mid)

        r02_used = r02_used_global

        # base diagnostics frame for this window
        diag = {
            "station": st,
            "w0": w0,
            "w1": w1,
            "mid": mid,
            "win_id": None,
            "r02_m": float(r02_used),
            "status": None,
            "reason": None,
            "n_cs2_window": np.nan,
            "n_cs2_radius": np.nan,
            "n_cs2_afterclip": np.nan,
            "n_gnss_in_win": np.nan,
        }

        if mid < POST_DATE_CUTOFF:
            diag["status"] = "rejected"
            diag["reason"] = "mid_before_cutoff"
            window_status_records.append(diag)
            continue

        # GNSS in-window
        gnss_in_win = gnss.loc[(gnss.index >= w0) & (gnss.index <= w1)]
        gnss_in_win = gnss_in_win[gnss_in_win["HGT_ADJUSTED(m)"].notna()]
        diag["n_gnss_in_win"] = int(len(gnss_in_win))

        has_left_gnss  = (gnss_in_win.index < mid).any()
        has_right_gnss = (gnss_in_win.index > mid).any()
        if not (has_left_gnss and has_right_gnss):
            diag["status"] = "rejected"
            diag["reason"] = "no_gnss_bracket"
            window_status_records.append(diag)
            continue

        # CS2 in [w0, w1)
        m_mask = (cs2_sta["time"] >= w0) & (cs2_sta["time"] < w1)
        cols_needed = ["cs2_row", "x", "y", "h_r1", "time", "sigma0_r1"]
        missing_cols = [c for c in cols_needed if c not in cs2_sta.columns]
        if missing_cols:
            raise ValueError(f"CS2 parquet missing columns: {missing_cols}")

        s_raw_window = cs2_sta.loc[m_mask, cols_needed].copy()
        diag["n_cs2_window"] = int(len(s_raw_window))
        if s_raw_window.empty:
            diag["status"] = "rejected"
            diag["reason"] = "no_cs2_in_window"
            window_status_records.append(diag)
            continue

        # GNSS-based station location for this window
        pos = window_avg_gnss_location(
            pos_df,
            t_center=mid,
            half_days=half_days,
            method="mean",
            valid_dates=gnss_in_win.index
        )
        if not pos.get("available", False):
            diag["status"] = "rejected"
            diag["reason"] = "no_gnss_position"
            window_status_records.append(diag)
            continue

        x_mid, y_mid = pos["sx_w"], pos["sy_w"]

        # spatial radius filter
        dx = s_raw_window["x"].to_numpy() - x_mid
        dy = s_raw_window["y"].to_numpy() - y_mid
        in_radius = (dx * dx + dy * dy) <= r02_used * r02_used
        s_raw = s_raw_window.loc[in_radius]
        diag["n_cs2_radius"] = int(len(s_raw))
        if len(s_raw) < 4:
            diag["status"] = "rejected"
            diag["reason"] = "lt4_cs2_in_radius"
            window_status_records.append(diag)
            continue

        # time bracketing for CS2
        has_left_cs2_time  = (s_raw["time"] < mid).any()
        has_right_cs2_time = (s_raw["time"] > mid).any()
        if not (has_left_cs2_time and has_right_cs2_time):
            diag["status"] = "rejected"
            diag["reason"] = "no_cs2_time_bracket"
            window_status_records.append(diag)
            continue

        # quadrants condition
        has_x_left  = (s_raw["x"] < x_mid).any()
        has_x_right = (s_raw["x"] > x_mid).any()
        has_y_below = (s_raw["y"] < y_mid).any()
        has_y_above = (s_raw["y"] > y_mid).any()
        if not (has_x_left and has_x_right and has_y_below and has_y_above):
            diag["status"] = "rejected"
            diag["reason"] = "quad_fail"
            window_status_records.append(diag)
            continue

        # window passes all pre-clip guards → candidate_preclip
        win_id = f"{st}_{mid.strftime('%Y%m%dT%H%M%S')}_R{int(r02_used)}m"
        diag["status"] = "candidate_preclip"
        diag["reason"] = "passes_all_preclip"
        diag["win_id"] = win_id

        win_status_index[win_id] = len(window_status_records)
        window_status_records.append(diag)

        # store in station-level pool
        station_pool_chunks.append(s_raw)

        # store window candidate (with cs2_row IDs only)
        window_candidates.append({
            "win_id": win_id,
            "station": st,
            "w0": w0,
            "w1": w1,
            "mid": mid,
            "x_mid": float(x_mid),
            "y_mid": float(y_mid),
            "r02_used_m": float(r02_used),
            "time_window_days": TIME_WINDOW_DAYS,
            "step_days": STEP_DAYS,
            "variogram": "fields_Matern",
            "cs2_rows": s_raw["cs2_row"].astype(int).tolist()
        })

    if not station_pool_chunks:
        print(f"  ⚠️ {st}: no windows passed geometry/time/GNSS guards; skipping station.")
        # still write diagnostics if any
        if window_status_records:
            diag_df = pd.DataFrame(window_status_records)
            diag_csv = os.path.join(st_root_dir, f"{st}_window_diagnostics.csv")
            diag_df.to_csv(diag_csv, index=False)
            print(f"  [DIAG] wrote {len(diag_df)} window diagnostics → {diag_csv}")
        continue

    station_pool = pd.concat(station_pool_chunks, ignore_index=True)
    station_pool = station_pool.drop_duplicates(subset=["cs2_row"]).reset_index(drop=True)
    print(f"  station-level pool size (unique CS2 rows) before clipping: {len(station_pool)}")

    # ---- PER-STATION ITERATIVE CLIPPING ----
    kept = station_pool.copy()
    for col in ("sigma0_r1", "h_r1"):
        if col not in kept.columns:
            raise ValueError(f"Column '{col}' missing in station_pool for {st}")
        kept = iterative_sigma_clip(kept, col=col, z=3.0, max_delta_pct=5.0)

    good_idx = set(kept["cs2_row"].astype(int).tolist())
    print(f"  station-level kept size after clipping: {len(good_idx)}")

    # ---- SECOND PASS: write per-window CSVs, maps & master metadata ----
    st_points_dir = safe_makedirs(os.path.join(WINDOW_POINTS_ROOT, st))
    winfig_dir_spatial   = safe_makedirs(os.path.join(st_root_dir, "window_point_maps"))
    winfig_dir_temporal  = safe_makedirs(os.path.join(st_root_dir, "temporal_point_maps"))

    kept_windows = 0
    for wc in window_candidates:
        cs2_ids = wc["cs2_rows"]
        win_id  = wc["win_id"]

        mask = station_pool["cs2_row"].astype(int).isin(cs2_ids) & \
               station_pool["cs2_row"].astype(int).isin(good_idx)
        s_win = station_pool.loc[mask, ["cs2_row", "x", "y", "h_r1", "time"]].copy()

        idx = win_status_index.get(win_id, None)

        if len(s_win) < 4:
            if idx is not None:
                window_status_records[idx]["status"] = "rejected_postclip"
                window_status_records[idx]["reason"] = "lt4_after_station_clip"
                window_status_records[idx]["n_cs2_afterclip"] = int(len(s_win))
            continue

        # this window survives → kept
        if idx is not None:
            window_status_records[idx]["status"] = "kept"
            window_status_records[idx]["reason"] = "used_for_kriging"
            window_status_records[idx]["n_cs2_afterclip"] = int(len(s_win))

        pts_csv = os.path.join(st_points_dir, f"{win_id}.csv")
        s_win.to_csv(pts_csv, index=False)

        # -------- per-window point maps (elevation-colored & temporal) --------
        try:
            mid_dt  = pd.to_datetime(wc["mid"])
            mid_str = mid_dt.strftime("%Y%m%d")
            x_mid   = wc["x_mid"]
            y_mid   = wc["y_mid"]
            r02_used = wc["r02_used_m"]

            # (A) Elevation-colored spatial map
            fig_map, axm = plt.subplots(figsize=(5.5, 5.2))
            sc = axm.scatter(s_win["x"], s_win["y"], c=s_win["h_r1"],
                             s=18, alpha=0.9)
            cb = plt.colorbar(sc, ax=axm)
            cb.set_label("CryoSat-2 elevation h_r1 (m)")

            axm.plot(x_mid, y_mid, marker="x", markersize=10, mew=2,
                     label="GNSS position")

            circ = Circle((x_mid, y_mid), r02_used,
                          fill=False, lw=1.2, alpha=0.7)
            axm.add_patch(circ)

            axm.set_aspect("equal", adjustable="box")
            axm.set_xlabel("X (m, EPSG:3031)")
            axm.set_ylabel("Y (m, EPSG:3031)")
            axm.set_title(f"{st} | {mid_dt.date()} | N={len(s_win)} | R={int(r02_used)} m")
            axm.grid(True, alpha=0.2)
            axm.legend(loc="best", frameon=True)

            out_map = os.path.join(
                winfig_dir_spatial,
                f"{st}_points_{mid_str}_R{int(r02_used)}m.png"
            )
            plt.tight_layout()
            plt.savefig(out_map, dpi=200)
            plt.close(fig_map)

            # (B) Temporal-colored spatial map
            tnums = mdates.date2num(pd.to_datetime(s_win["time"]))
            fig_t, axt = plt.subplots(figsize=(5.5, 5.2))
            sc_t = axt.scatter(s_win["x"], s_win["y"], c=tnums,
                               s=22, alpha=0.95, cmap="viridis")
            cbt = plt.colorbar(sc_t, ax=axt)
            cbt.ax.set_ylabel("Acquisition date")

            locator = mdates.AutoDateLocator(minticks=4, maxticks=8)
            formatter = mdates.ConciseDateFormatter(locator)
            cbt.ax.yaxis.set_major_locator(locator)
            cbt.ax.yaxis.set_major_formatter(formatter)

            axt.plot(x_mid, y_mid, marker="x", markersize=10, mew=2,
                     color="k", label="GNSS position")

            circ2 = Circle((x_mid, y_mid), r02_used,
                           fill=False, lw=1.2, alpha=0.7)
            axt.add_patch(circ2)

            axt.set_aspect("equal", adjustable="box")
            axt.set_xlabel("X (m, EPSG:3031)")
            axt.set_ylabel("Y (m, EPSG:3031)")
            tmin_str = pd.to_datetime(s_win["time"].min()).date()
            tmax_str = pd.to_datetime(s_win["time"].max()).date()
            axt.set_title(
                f"{st} | {mid_dt.date()} | N={len(s_win)} | R={int(r02_used)} m\n"
                f"CS2 span: {tmin_str} → {tmax_str}"
            )
            axt.grid(True, alpha=0.2)
            axt.legend(loc="best", frameon=True)

            out_map_t = os.path.join(
                winfig_dir_temporal,
                f"{st}_points_TEMPORAL_{mid_str}_R{int(r02_used)}m.png"
            )
            plt.tight_layout()
            plt.savefig(out_map_t, dpi=200)
            plt.close(fig_t)
        except Exception as e:
            print(f"⚠️ {st} {wc['mid']}: failed to save window maps ({e})")

        # -------- master record --------
        master_records.append({
            "window_id": wc["win_id"],
            "station": wc["station"],
            "w0": wc["w0"],
            "w1": wc["w1"],
            "mid": wc["mid"],
            "x_mid": wc["x_mid"],
            "y_mid": wc["y_mid"],
            "r02_m": wc["r02_used_m"],
            "time_window_days": wc["time_window_days"],
            "step_days": wc["step_days"],
            "variogram": wc["variogram"],
            "n_points": int(len(s_win)),
            "points_csv": pts_csv
        })
        kept_windows += 1

    print(f"  {st}: kept {kept_windows} windows after clipping/point-count filter.")

    # -------- write per-station diagnostics CSV --------
    if window_status_records:
        diag_df = pd.DataFrame(window_status_records)
        diag_csv = os.path.join(st_root_dir, f"{st}_window_diagnostics.csv")
        diag_df.to_csv(diag_csv, index=False)
        print(f"  [DIAG] wrote {len(diag_df)} window diagnostics → {diag_csv}")



## Write the master table


In [ ]:
# ---------------------------
# WRITE MASTER CSV
# ---------------------------

if not master_records:
    raise RuntimeError("No windows survived per-station clipping; master_records is empty.")

master_df = pd.DataFrame(master_records)
master_df = master_df.sort_values(["station", "mid"]).reset_index(drop=True)

KRIG_META_CSV = os.path.join(OUTPUT_DIR, "kriging_windows_master.csv")
master_df.to_csv(KRIG_META_CSV, index=False)
print(f"\n[MASTER] wrote {len(master_df)} windows → {KRIG_META_CSV}")
